In [1]:
import duckdb
import pandas as pd

# Connexion DuckDB
con = duckdb.connect()

# Charger tous les CSV en tables
con.execute("CREATE TABLE customers AS SELECT * FROM read_csv_auto('olist_customers_dataset.csv')")
con.execute("CREATE TABLE geolocation AS SELECT * FROM read_csv_auto('olist_geolocation_dataset.csv')")
con.execute("CREATE TABLE orders AS SELECT * FROM read_csv_auto('olist_orders_dataset.csv')")
con.execute("CREATE TABLE order_items AS SELECT * FROM read_csv_auto('olist_order_items_dataset.csv')")
con.execute("CREATE TABLE order_payments AS SELECT * FROM read_csv_auto('olist_order_payments_dataset.csv')")
con.execute("CREATE TABLE order_reviews AS SELECT * FROM read_csv_auto('olist_order_reviews_dataset.csv')")
con.execute("CREATE TABLE products AS SELECT * FROM read_csv_auto('olist_products_dataset.csv')")
con.execute("CREATE TABLE sellers AS SELECT * FROM read_csv_auto('olist_sellers_dataset.csv')")
con.execute("CREATE TABLE category_translation AS SELECT * FROM read_csv_auto('product_category_name_translation.csv')")

print("Tables chargées ✅")

Tables chargées ✅


In [2]:
# Vérifier que toutes les tables sont là
print("=== TABLES DISPONIBLES ===")
print(con.execute("SHOW TABLES").df())

print("\n=== APERÇU ORDERS ===")
print(con.execute("SELECT * FROM orders LIMIT 3").df())

print("\n=== APERÇU ORDER_ITEMS ===")
print(con.execute("SELECT * FROM order_items LIMIT 3").df())

print("\n=== COLONNES DE CHAQUE TABLE ===")
tables = ['orders', 'order_items', 'order_payments', 'order_reviews', 
          'products', 'sellers', 'customers', 'category_translation']
for t in tables:
    cols = con.execute(f"DESCRIBE {t}").df()['column_name'].tolist()
    print(f"{t}: {cols}")

=== TABLES DISPONIBLES ===
                   name
0  category_translation
1             customers
2           geolocation
3           order_items
4        order_payments
5         order_reviews
6                orders
7              products
8               sellers

=== APERÇU ORDERS ===
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00      

## Q1 — Délai moyen de livraison par état
# Question business :** Quels états brésiliens ont les pires délais de livraison ?


In [ ]:

q1 = con.execute("""
    SELECT 
        c.customer_state,
        ROUND(AVG(
            DATEDIFF('day', 
                o.order_purchase_timestamp, 
                o.order_delivered_customer_date)
        ), 1) AS avg_delivery_days,
        COUNT(*) AS total_orders
    FROM orders o
    JOIN customers c USING (customer_id)
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY c.customer_state
    ORDER BY avg_delivery_days DESC
""").df()

q1

,customer_state,avg_delivery_days,total_orders
0,RR,29.3,41
1,AP,27.2,67
2,AM,26.4,145
3,AL,24.5,397
4,PA,23.7,946
5,MA,21.5,717
6,SE,21.5,335
7,CE,21.2,1279
8,AC,21.0,80
9,PB,20.4,517


**Insight :** Les états du nord (RR, AP, AM) attendent en moyenne 29 jours 
contre 8.7 jours pour São Paulo — soit 3x plus longtemps. Ces 3 états 
représentent seulement 253 commandes au total, ce qui suggère que la faible 
densité logistique explique ces délais. Une stratégie de stocks décentralisés 
dans le nord réduirait significativement ce gap.

## Q2 — Top 10 vendeurs par chiffre d'affaires
#Question business :** Quels vendeurs génèrent le plus de revenus sur la plateforme ?

In [ ]:

q2 = con.execute("""
    SELECT 
        s.seller_id,
        s.seller_state,
        ROUND(SUM(oi.price), 2) AS total_revenue,
        COUNT(DISTINCT oi.order_id) AS total_orders,
        ROUND(AVG(oi.price), 2) AS avg_order_value
    FROM order_items oi
    JOIN sellers s USING (seller_id)
    GROUP BY s.seller_id, s.seller_state
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

q2

,seller_id,seller_state,total_revenue,total_orders,avg_order_value
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,229472.63,1132,198.51
1,53243585a1d6dc2643021fd1853d8905,BA,222776.05,358,543.36
2,4a3ca9315b744ce9f8e9374361493884,SP,200472.92,1806,100.89
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,194042.03,585,331.13
4,7c67e1448b00f6e969d365cea6b010ab,SP,187923.89,982,137.77
5,7e93a43ef30c4f03f38b393420bc753a,SP,176431.87,336,518.92
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,160236.57,1314,103.31
7,7a67c85e85bb2ce8582c35f2203ad736,SP,141745.53,1160,121.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,138968.55,915,97.32
9,955fee9216a65b617aa5c0531780ce60,SP,135171.70,1287,90.17


**Insight :** 9 des 10 meilleurs vendeurs sont basés à São Paulo — seul 
le #2 vient de Bahia, avec un panier moyen de 543€ contre 100-200€ pour 
les vendeurs SP. Deux stratégies distinctes émergent : volume élevé + 
prix bas (SP) vs volume faible + prix premium (BA). La plateforme pourrait 
encourager les vendeurs premium hors SP pour diversifier géographiquement 
ses revenus.

In [8]:
## Q3 — Taux de commandes livrées en retard
# Question business :** Quelle proportion des commandes n'est pas livrée dans les délais estimés ?

q3 = con.execute("""
    SELECT
        COUNT(*) AS total_delivered,
        SUM(CASE 
            WHEN order_delivered_customer_date > order_estimated_delivery_date 
            THEN 1 ELSE 0 
        END) AS late_orders,
        ROUND(
            100.0 * SUM(CASE 
                WHEN order_delivered_customer_date > order_estimated_delivery_date 
                THEN 1 ELSE 0 
            END) / COUNT(*), 2
        ) AS late_rate_pct,
        ROUND(AVG(
            DATEDIFF('day',
                order_estimated_delivery_date,
                order_delivered_customer_date)
        ), 1) AS avg_days_late
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
      AND order_estimated_delivery_date IS NOT NULL
""").df()

q3

,total_delivered,late_orders,late_rate_pct,avg_days_late
0,96470,7826.0,8.11,-11.9


**Insight :** 8.11% des commandes sont livrées en retard (7 826 sur 96 470), 
mais la moyenne globale est de -11.9 jours — ce qui signifie que la majorité 
des commandes arrivent presque 12 jours AVANT la date estimée. Olist 
sur-estime volontairement les délais pour gérer les attentes clients. 
Le vrai problème se concentre sur les 8% en retard, probablement corrélés 
aux états du nord vus en Q1.

In [9]:
## Q4 — Catégories de produits avec le meilleur score d'avis
## Question business :** Quelles catégories satisfont le plus les clients ?

q4 = con.execute("""
    SELECT
        ct.product_category_name_english AS category,
        ROUND(AVG(r.review_score), 2) AS avg_score,
        COUNT(*) AS total_reviews
    FROM order_reviews r
    JOIN order_items oi USING (order_id)
    JOIN products p USING (product_id)
    JOIN category_translation ct USING (product_category_name)
    GROUP BY category
    HAVING COUNT(*) > 100
    ORDER BY avg_score DESC
    LIMIT 10
""").df()

q4

,category,avg_score,total_reviews
0,books_general_interest,4.45,549
1,books_technical,4.37,266
2,luggage_accessories,4.32,1088
3,food_drink,4.32,279
4,fashion_shoes,4.23,261
5,food,4.22,495
6,pet_shop,4.19,1939
7,stationery,4.19,2507
8,computers,4.18,200
9,home_appliances,4.17,806


**Insight :** Les livres et accessoires de voyage dominent la satisfaction 
client (4.45 et 4.32/5). Ce sont des produits simples à expédier, légers 
et peu sujets aux dommages — la qualité logistique explique probablement 
ces scores élevés autant que le produit lui-même. À l'inverse, les 
catégories absentes du top 10 (électronique, meubles) méritent 
une investigation séparée.

In [10]:
## Q5 — Évolution mensuelle du chiffre d'affaires
## Question business :** Quelle est la croissance du CA mois par mois ?

q5 = con.execute("""
    WITH monthly_revenue AS (
        SELECT
            STRFTIME(o.order_purchase_timestamp, '%Y-%m') AS month,
            ROUND(SUM(oi.price), 2) AS revenue
        FROM orders o
        JOIN order_items oi USING (order_id)
        WHERE o.order_status != 'canceled'
        GROUP BY month
        ORDER BY month
    )
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
        ROUND(
            100.0 * (revenue - LAG(revenue) OVER (ORDER BY month)) 
            / LAG(revenue) OVER (ORDER BY month), 1
        ) AS growth_pct
    FROM monthly_revenue
    ORDER BY month
""").df()

q5

,month,revenue,prev_month_revenue,growth_pct
0,2016-09,207.86,NaN,NaN
1,2016-10,46514.99,207.86,22278.0
2,2016-12,10.90,46514.99,-100.0
3,2017-01,120098.27,10.90,1101719.0
4,2017-02,244959.35,120098.27,104.0
5,2017-03,368341.32,244959.35,50.4
6,2017-04,353842.98,368341.32,-3.9
7,2017-05,503159.19,353842.98,42.2
8,2017-06,429916.61,503159.19,-14.6
9,2017-07,492287.30,429916.61,14.5


**Insight :** Le CA passe de ~46k€ en octobre 2016 à ~993k€ en avril 2018 
— multiplication par 21 en 18 mois. Deux anomalies à noter : novembre 2017 
(+52% — Black Friday) et les mois 2016-09 et 2018-09 qui affichent des 
valeurs quasi nulles (données incomplètes, mois non terminés dans le dataset). 
La croissance se stabilise début 2018 autour de 900k-1M€/mois, signal de 
maturité de la plateforme.

## Q6 — Panier moyen par méthode de paiement
**Question business :** Les clients paient-ils plus selon leur 
méthode de paiement ?

In [11]:
q6 = con.execute("""
    SELECT
        payment_type,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(AVG(payment_value), 2) AS avg_order_value,
        ROUND(SUM(payment_value), 2) AS total_revenue,
        ROUND(100.0 * COUNT(DISTINCT order_id) / 
            SUM(COUNT(DISTINCT order_id)) OVER (), 1) AS pct_of_orders
    FROM order_payments
    GROUP BY payment_type
    ORDER BY total_revenue DESC
""").df()

q6

,payment_type,total_orders,avg_order_value,total_revenue,pct_of_orders
0,credit_card,76505,163.32,12542084.19,75.2
1,boleto,19784,145.03,2869361.27,19.5
2,voucher,3866,65.70,379436.87,3.8
3,debit_card,1528,142.57,217989.79,1.5
4,not_defined,3,0.00,0.00,0.0


**Insight :** La carte de crédit domine massivement — 75% des commandes 
et panier moyen le plus élevé (163€ vs 145€ pour le boleto). Les vouchers 
représentent 3.8% des commandes mais avec un panier 2x plus faible (65€) — 
ils sont probablement utilisés pour des petits achats ou en complément 
d'un autre paiement. Le débit card est quasi inexistant (1.5%) malgré 
un panier similaire au boleto.

## Q7 — Clients fidèles vs clients uniques
**Question business :** Les clients qui commandent plusieurs fois 
représentent-ils une part significative du CA ?

In [12]:
q7 = con.execute("""
    WITH customer_orders AS (
        SELECT
            c.customer_unique_id,
            COUNT(DISTINCT o.order_id) AS order_count,
            ROUND(SUM(oi.price), 2) AS total_spent
        FROM orders o
        JOIN customers c USING (customer_id)
        JOIN order_items oi USING (order_id)
        WHERE o.order_status != 'canceled'
        GROUP BY c.customer_unique_id
    )
    SELECT
        CASE 
            WHEN order_count = 1 THEN 'Single order'
            WHEN order_count = 2 THEN '2 orders'
            ELSE '3+ orders'
        END AS customer_type,
        COUNT(*) AS customer_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_customers,
        ROUND(SUM(total_spent), 2) AS total_revenue,
        ROUND(100.0 * SUM(total_spent) / SUM(SUM(total_spent)) OVER (), 1) AS pct_revenue,
        ROUND(AVG(total_spent), 2) AS avg_spent_per_customer
    FROM customer_orders
    GROUP BY customer_type
    ORDER BY customer_count DESC
""").df()

q7

,customer_type,customer_count,pct_customers,total_revenue,pct_revenue,avg_spent_per_customer
0,Single order,92102,97.0,12745940.00,94.4,138.39
1,2 orders,2651,2.8,650963.95,4.8,245.55
2,3+ orders,236,0.2,99504.48,0.7,421.63


**Insight :** 97% des clients ne commandent qu'une seule fois — Olist a un 
problème de rétention massif. Pourtant les clients fidèles dépensent 
en moyenne 3x plus (421€ pour 3+ commandes vs 138€ pour une seule). 
Un programme de fidélisation ciblant même 5% des clients single-order 
pourrait générer des millions de CA additionnel sans acquisition 
de nouveaux clients.

## Q8 — Vendeurs avec un taux d'avis négatifs élevé
**Question business :** Quels vendeurs concentrent le plus 
de mauvaises expériences client ?

In [13]:
q8 = con.execute("""
    WITH seller_reviews AS (
        SELECT
            s.seller_id,
            s.seller_state,
            COUNT(*) AS total_reviews,
            SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END) AS negative_reviews,
            ROUND(100.0 * SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END) 
                / COUNT(*), 1) AS negative_rate_pct,
            ROUND(AVG(r.review_score), 2) AS avg_score
        FROM order_reviews r
        JOIN order_items oi USING (order_id)
        JOIN sellers s USING (seller_id)
        GROUP BY s.seller_id, s.seller_state
        HAVING COUNT(*) > 30
    )
    SELECT *
    FROM seller_reviews
    WHERE negative_rate_pct > 20
    ORDER BY negative_rate_pct DESC
    LIMIT 10
""").df()

q8

,seller_id,seller_state,total_reviews,negative_reviews,negative_rate_pct,avg_score
0,1ca7077d890b907f89be8c954a02686a,SP,136,89.0,65.4,2.20
1,2709af9587499e95e803a6498a5a56e9,SP,46,27.0,58.7,2.57
2,bb135baca94c82fcb731335ad5b04a03,SP,38,20.0,52.6,2.66
3,ec4608a1f76453166bb312b2968aeaf4,SP,35,18.0,51.4,2.69
4,2eb70248d66e0e3ef83659f71b244378,SP,209,105.0,50.2,2.71
5,602044f2c16190c2c6e45eb35c2e21cb,SP,59,27.0,45.8,2.93
6,54965bbe3e4f07ae045b90b0b8541f52,PR,81,37.0,45.7,2.94
7,95f83f51203c626648c875dd41874c7f,MG,68,31.0,45.6,3.18
8,bbad7e518d7af88a0897397ffdca1979,SP,84,38.0,45.2,3.04
9,8e6d7754bc7e0f22c96d255ebda59eba,SP,133,59.0,44.4,3.02


**Insight :** Le vendeur #1 a un taux d'avis négatifs de 65.4% sur 136 avis 
— il devrait être suspendu. Les 10 pires vendeurs sont quasi exclusivement 
basés à SP, ce qui est logique vu que SP concentre la majorité des vendeurs. 
Un système d'alerte automatique à 20% de taux négatif permettrait 
d'intervenir avant que ces vendeurs dégradent la note globale 
de la plateforme.

## Q9 — Prix du produit vs score d'avis
**Question business :** Les produits plus chers reçoivent-ils 
de meilleurs avis ?

In [16]:
q9 = con.execute("""
    SELECT
        CASE
            WHEN oi.price < 50   THEN '< 50€'
            WHEN oi.price < 100  THEN '50-100€'
            WHEN oi.price < 200  THEN '100-200€'
            WHEN oi.price < 500  THEN '200-500€'
            ELSE '500€+'
        END AS price_bucket,
        COUNT(*) AS total_orders,
        ROUND(AVG(r.review_score), 2) AS avg_score,
        ROUND(AVG(oi.price), 2) AS avg_price
    FROM order_items oi
    JOIN order_reviews r USING (order_id)
    GROUP BY price_bucket
    ORDER BY avg_price
""").df()

q9

,price_bucket,total_orders,avg_score,avg_price
0,< 50€,38959,4.03,31.31
1,50-100€,33095,4.02,74.78
2,100-200€,26908,4.06,143.14
3,200-500€,10202,4.03,296.17
4,500€+,3208,4.00,922.52


**Insight :** Contre-intuitivement, le prix n'a quasiment aucun impact 
sur la satisfaction client — les scores varient entre 4.00 et 4.06 
quelle que soit la tranche de prix. Un client qui paie 922€ est aussi 
satisfait qu'un client qui paie 31€. Cela suggère que la satisfaction 
est pilotée par la logistique et le service (délai, état du colis) 
plutôt que par le produit lui-même.

## Q10 — Top 5 catégories par CA avec part du total
**Question business :** Quelles catégories génèrent le plus de revenus 
et quelle est leur part du CA total ?

In [17]:
q10 = con.execute("""
    WITH category_revenue AS (
        SELECT
            ct.product_category_name_english AS category,
            ROUND(SUM(oi.price), 2) AS revenue,
            COUNT(DISTINCT oi.order_id) AS total_orders
        FROM order_items oi
        JOIN products p USING (product_id)
        JOIN category_translation ct USING (product_category_name)
        GROUP BY category
    )
    SELECT
        category,
        revenue,
        total_orders,
        ROUND(100.0 * revenue / SUM(revenue) OVER (), 2) AS pct_of_total_revenue,
        RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
    FROM category_revenue
    ORDER BY revenue DESC
    LIMIT 5
""").df()

q10

,category,revenue,total_orders,pct_of_total_revenue,revenue_rank
0,health_beauty,1258681.34,8836,9.39,1
1,watches_gifts,1205005.68,5624,8.99,2
2,bed_bath_table,1036988.68,9417,7.73,3
3,sports_leisure,988048.97,7720,7.37,4
4,computers_accessories,911954.32,6689,6.80,5


**Insight :** Les 5 premières catégories représentent 40.28% du CA total 
de la plateforme. Health & Beauty domine avec 9.39% malgré seulement 
8 836 commandes — un panier moyen élevé. À l'inverse, Bed & Bath génère 
plus de commandes (9 417) mais moins de CA, signe d'un panier plus faible. 
Une stratégie de croissance ciblant Health & Beauty et Watches & Gifts 
maximiserait le CA sans nécessairement augmenter le volume de commandes.